# Shelf List Report

This notebook will create a CSV file of Inventory Items with the following information: Barcode, Title, Effective Location, Effective Call Number Components, Material Type, and Item Status

## 1. Environment setup

In [26]:
# This script will import the following Python libraries, but not install them. If you are missing any of these, you can install them via the command line like this:
# !pip install pandas
import pandas as pd
import requests
from datetime import datetime, timedelta   

pd.set_option('display.max_columns', None)

## 2. Log in from a shared notebook

This is the notebook that will authenticate you to the FOLIO API and store your access token in a variable called ACCESS_TOKEN. You will need to run this cell before running any other cells in this notebook.

In [27]:
%run folio_auth.ipynb

Login succeeded. Token retrieved.


## 3. Helper for paginated GET requests

FOLIO endpoints typically page results via `limit`/`offset` query params and return a
JSON object with a records array plus a total count. This helper loops until it has
everything. 

In [28]:
def fetch_all_records(endpoint, records_key, limit=1000, query=None):
    all_records = []
    offset = 0
    base_url = OKAPI_URL
    headers = HEADERS.copy() if 'HEADERS' in globals() else {"X-Okapi-Tenant": TENANT, "Content-Type": "application/json"}
    if 'token' in globals():
        headers["Authorization"] = f"Bearer {token}"

    while True:
        params = {"limit": limit, "offset": offset}
        if query:
            params["query"] = query
        response = session.get(f"{base_url}{endpoint}", headers=headers, params=params)
        response.raise_for_status()
        payload = response.json()

        batch = payload.get(records_key, [])
        all_records.extend(batch)

        if len(batch) < limit:
            break
        offset += limit
    print(response.url)
    return all_records

## 4. Get Locations

In [29]:


locs_raw = fetch_all_records(
    "/locations",
    records_key="locations",
    limit=100,
    query='isActive=="true"',
) 

print(f"{len(locs_raw)} active locations found")
locs_df = pd.DataFrame(locations_raw)
print(locs_df.loc[:, ['name','id']])


https://api-demo.folio.ebsco.com/locations?limit=100&offset=0&query=isActive%3D%3D%22true%22
27 active locations found
                                  name                                    id
0               Music Library Reserves  757c6f19-bf53-4c63-b314-135e7360afc0
1                   Interlibrary Loans  b8b8dcd5-69f3-4165-909e-435e8b9cdd2e
2                      Suggestion Form  923b821c-a192-43b6-b1a7-8e03f50d1086
3     Main Library Special Collections  8ee52ff5-d149-48bc-bf4f-749a0097325e
4                      Law Main Stacks  e3775cef-9d30-4896-a452-98a099cf9bfa
5      Main Library Technical Services  d9d54cc4-b052-42b8-8dd2-0f192752d88d
6                    MC Alton Reserves  4df811f6-2c1b-4690-aa08-66a12d5f3bf3
7                   MC Alton Reference  7bd4f123-cacd-458b-acbb-458a82f9f34c
8                 MC Alton Periodicals  ecc6b76e-4b6a-49de-8378-d7edd8ba3e20
9                          Bus Lib Ref  aa902fd1-ca2e-4d06-89ab-96f79e1feee0
10                     MC Alton St

## 5. Retrieve Item records with that Effective Location

In [ ]:
location_id = 'fc4b17f0-3745-4827-8e74-08f004003f7d'
# location_id = 'REPLACE WITH THE LOCATION ID FROM ABOVE'

items_raw = fetch_all_records(
    "/item-storage/items",
    records_key="items",
    query='limit=0&query=(effectiveLocationId=='+location_id+')'
) 
location_name = locations_df[locations_df['id']==location_id]['name'].values[0]
print(f"{len(items_raw)} items found for with an effective location of: {location_name}")
items_df = pd.DataFrame(items_raw)


https://api-demo.folio.ebsco.com/item-storage/items?limit=1000&offset=0&query=limit%3D0%26query%3D%28effectiveLocationId%3D%3Dfc4b17f0-3745-4827-8e74-08f004003f7d%29
10 items found for with an effective location of: MED Director's Office


## 6. Get other Material Types for merge

In [ ]:
mats_raw = fetch_all_records(
    "/material-types",
    records_key="mtypes",
    limit=99
)
print(f"{len(mats_raw)} material types found")
mats_df = pd.DataFrame(mats_raw)

https://api-demo.folio.ebsco.com/material-types?limit=99&offset=0
16 material types found
